In [4]:
import pandas as pd

df = pd.read_csv('../data/processed/games_clean.csv')

df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])

print(df.dtypes)

SEASON_ID                     int64
TEAM_ID                       int64
TEAM_ABBREVIATION               str
TEAM_NAME                       str
GAME_ID                       int64
GAME_DATE            datetime64[us]
MATCHUP                         str
WL                              str
MIN                           int64
PTS                           int64
FGM                           int64
FGA                           int64
FG_PCT                      float64
FG3M                          int64
FG3A                          int64
FG3_PCT                     float64
FTM                           int64
FTA                           int64
FT_PCT                      float64
OREB                          int64
DREB                          int64
REB                           int64
AST                           int64
STL                           int64
BLK                           int64
TOV                           int64
PF                            int64
PLUS_MINUS                  

Rest days and back to back

In [5]:
df_sorted = df.sort_values(['TEAM_ABBREVIATION', 'GAME_DATE'])

df_sorted['rest_days'] = df_sorted.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])['GAME_DATE'].diff().dt.days

median_rest = df_sorted['rest_days'].median()
df_sorted['rest_days'] = df_sorted['rest_days'].fillna(median_rest)

df_sorted['is_back_to_back'] = (df_sorted['rest_days'] == 1).astype(int)

print(df_sorted[['TEAM_ABBREVIATION', 'SEASON_ID', 'GAME_DATE', 'rest_days', 'is_back_to_back']].head(15))

     TEAM_ABBREVIATION  SEASON_ID  GAME_DATE  rest_days  is_back_to_back
2138               ATL      22020 2020-12-23        2.0                0
2110               ATL      22020 2020-12-26        3.0                0
2074               ATL      22020 2020-12-28        2.0                0
2047               ATL      22020 2020-12-30        2.0                0
2007               ATL      22020 2021-01-01        2.0                0
2004               ATL      22020 2021-01-02        1.0                1
1968               ATL      22020 2021-01-04        2.0                0
1947               ATL      22020 2021-01-06        2.0                0
1882               ATL      22020 2021-01-09        3.0                0
1861               ATL      22020 2021-01-11        2.0                0
1809               ATL      22020 2021-01-15        4.0                0
1798               ATL      22020 2021-01-16        1.0                1
1779               ATL      22020 2021-01-18       

In [6]:
print(df_sorted.value_counts('is_back_to_back'))

is_back_to_back
0    11866
1     2594
Name: count, dtype: int64


In [7]:
print(df_sorted[(df_sorted['TEAM_ABBREVIATION'] == 'ATL')].groupby('SEASON_ID').head(1)[
    ['SEASON_ID', 'GAME_DATE', 'rest_days']])

       SEASON_ID  GAME_DATE  rest_days
2138       22020 2020-12-23        2.0
4593       22021 2021-10-21        2.0
7073       22022 2022-10-19        2.0
9533       22023 2023-10-25        2.0
11986      22024 2024-10-23        2.0
14437      22025 2025-10-22        2.0


Rolling averages

In [8]:
stat_cols = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FG_PCT', 'FG3_PCT']

window=5

for col in stat_cols:
    df_sorted[f'rolling_{col.lower()}_{window}'] = df_sorted.groupby(['TEAM_ABBREVIATION', 'SEASON_ID'])[col].shift(1).rolling(window=window, min_periods=1).mean()



In [9]:
print(df_sorted[['TEAM_ABBREVIATION', 'PTS', 'rolling_pts_5']].head(15))

     TEAM_ABBREVIATION  PTS  rolling_pts_5
2138               ATL  124            NaN
2110               ATL  122     124.000000
2074               ATL  128     123.000000
2047               ATL  141     124.666667
2007               ATL  114     128.750000
2004               ATL   91     125.800000
1968               ATL  108     119.200000
1947               ATL   94     116.400000
1882               ATL  105     109.600000
1861               ATL  112     102.400000
1809               ATL   92     102.000000
1798               ATL  106     102.200000
1779               ATL  108     101.800000
1747               ATL  123     104.600000
1731               ATL  116     108.200000


Elo

In [10]:
def calculate_elo(matched_df, k=20, home_advantage=100, initial_elo=1500, season_regression=0.75):
    # matched_df: one row one match 
 
    ratings = {}  # dict: name of the team : current ranking
    home_elo_before = []  # rating before match
    away_elo_before = []
    current_season = None
 
    for idx, row in matched_df.iterrows():
        team_home = row['HOME_TEAM_ABBR']
        team_away = row['AWAY_TEAM_ABBR']
        season = row['SEASON_ID']
 
        # 1. season changed 
        if current_season is not None and season != current_season:
            for team in ratings:
                ratings[team] = season_regression * ratings[team] + (1 - season_regression) * initial_elo
        current_season = season
 
        # 2. load current rating (initial 1500)
        home_rating = ratings.get(team_home, initial_elo)
        away_rating = ratings.get(team_away, initial_elo)
 
        # 3. save rating
        home_elo_before.append(home_rating)
        away_elo_before.append(away_rating)
 
        # 4. expected score (with home bonus)
        expected_home = 1 / (1 + 10 ** ((away_rating - (home_rating + home_advantage)) / 400))
 
        # 5. actual score
        actual_home = row['home_win']
 
        # 6. updating ratings after match
        new_home_rating = home_rating + k * (actual_home - expected_home)
        new_away_rating = away_rating + k * ((1 - actual_home) - (1 - expected_home))
 
        ratings[team_home] = new_home_rating
        ratings[team_away] = new_away_rating
 
    matched_df['home_elo'] = home_elo_before
    matched_df['away_elo'] = away_elo_before
    return matched_df

In [11]:
matched = pd.read_csv("../data/processed/games_matched.csv")

matched = calculate_elo(matched)

print(matched[['GAME_DATE', 'HOME_TEAM_ABBR', 'AWAY_TEAM_ABBR', 'home_elo', 'away_elo', 'home_win']].head(30))

     GAME_DATE HOME_TEAM_ABBR AWAY_TEAM_ABBR     home_elo   away_elo  home_win
0   2020-12-22            LAL            LAC  1500.000000  1500.0000         0
1   2020-12-22            BKN            GSW  1500.000000  1500.0000         1
2   2020-12-23            DEN            SAC  1500.000000  1500.0000         0
3   2020-12-23            IND            NYK  1500.000000  1500.0000         1
4   2020-12-23            TOR            NOP  1500.000000  1500.0000         0
5   2020-12-23            PHX            DAL  1500.000000  1500.0000         1
6   2020-12-23            PHI            WAS  1500.000000  1500.0000         1
7   2020-12-23            MEM            SAS  1500.000000  1500.0000         0
8   2020-12-23            BOS            MIL  1500.000000  1500.0000         1
9   2020-12-23            CHI            ATL  1500.000000  1500.0000         0
10  2020-12-23            CLE            CHA  1500.000000  1500.0000         1
11  2020-12-23            MIN            DET  1500.0

In [12]:
team = 'BOS'
team_games = matched[(matched['HOME_TEAM_ABBR'] == team) | (matched['AWAY_TEAM_ABBR'] == team)].copy()

team_games['team_elo'] = team_games.apply(
    lambda row: row['home_elo'] if row['HOME_TEAM_ABBR'] == team else row['away_elo'],
    axis=1
)

print(team_games.groupby('SEASON_ID').tail(3)[['GAME_DATE', 'SEASON_ID', 'team_elo']])
print("---")
print(team_games.groupby('SEASON_ID').head(3)[['GAME_DATE', 'SEASON_ID', 'team_elo']])

       GAME_DATE  SEASON_ID     team_elo
1039  2021-05-12      22020  1484.395477
1062  2021-05-15      22020  1472.752658
1071  2021-05-16      22020  1483.997660
2269  2022-04-06      22021  1612.407026
2280  2022-04-07      22021  1622.406408
2307  2022-04-10      22021  1614.765002
3504  2023-04-05      22022  1630.556259
3513  2023-04-07      22022  1635.448055
3539  2023-04-09      22022  1640.134739
4738  2024-04-11      22023  1722.465773
4744  2024-04-12      22023  1706.280424
4761  2024-04-14      22023  1707.417335
5959  2025-04-09      22024  1712.584917
5976  2025-04-11      22024  1698.866888
5996  2025-04-13      22024  1699.758097
7199  2026-04-09      22025  1680.685807
7200  2026-04-10      22025  1671.938094
7226  2026-04-12      22025  1673.902686
---
       GAME_DATE  SEASON_ID     team_elo
8     2020-12-23      22020  1500.000000
18    2020-12-25      22020  1507.198700
36    2020-12-27      22020  1494.397400
1091  2021-10-20      22021  1484.206101
1104  2021-1

Streak

In [13]:
def add_streak(df):
    df = df.sort_values(['TEAM_ABBREVIATION', 'SEASON_ID', 'GAME_DATE']).reset_index(drop=True)

    streak_before_game = []
    current_streak = 0
    current_team = 0
    current_season = None

    for idx, row in df.iterrows():
        team = row['TEAM_ABBREVIATION']
        season = row['SEASON_ID']
        if team != current_team or season != current_season:
            current_streak = 0

        streak_before_game.append(current_streak)

        if row['WL'] == 'W':
            if current_streak >= 0:
                current_streak += 1
            else:
                current_streak = 1
        elif row['WL'] == 'L':
            if current_streak <= 0:
                current_streak -= 1
            else:
                current_streak = -1

        current_team = team
        current_season = season

        pass

    df['streak'] = streak_before_game
    return df
    

In [16]:
df_sorted = add_streak(df_sorted)

print(df_sorted[['TEAM_ABBREVIATION', 'WL', 'streak']].head(15))

   TEAM_ABBREVIATION WL  streak
0                ATL  W       0
1                ATL  W       1
2                ATL  W       2
3                ATL  L       3
4                ATL  W      -1
5                ATL  L       1
6                ATL  L      -1
7                ATL  L      -2
8                ATL  L      -3
9                ATL  W      -4
10               ATL  L       1
11               ATL  L      -1
12               ATL  W      -2
13               ATL  W       1
14               ATL  W       2
